In [1]:
#initial audit
import pandas as pd
from pathlib import Path

# Dataset locations
PROJECT_DIR = Path.home() / "Climate-Change-Awareness-Dashboard"
INTERIM_DIR = PROJECT_DIR / "precipitation" / "data" / "interim"

# Modelling period
START_YEAR = 1951
END_YEAR = 2025

audit_results = []

for year in range(START_YEAR, END_YEAR + 1):

    file_path = INTERIM_DIR / f"imd_rainfall_{year}.csv"

    if not file_path.exists():
        print(f"Missing file: {year}")
        continue

    total_rows = 0
    null_values = 0
    negative_values = 0
    min_rainfall = float("inf")
    max_rainfall = float("-inf")
    daily_counts = pd.Series(dtype="int64")

    # Read in chunks to control memory usage
    for chunk in pd.read_csv(file_path, parse_dates=["date"], chunksize=500_000):

        total_rows += len(chunk)

        null_values += chunk.isna().sum().sum()

        negative_values += (chunk["rainfall_mm"].dropna() < 0).sum()

        if chunk["rainfall_mm"].notna().any():
            min_rainfall = min(min_rainfall, chunk["rainfall_mm"].min())
            max_rainfall = max(max_rainfall, chunk["rainfall_mm"].max())

        daily_counts = daily_counts.add(
            chunk["date"].value_counts(),
            fill_value=0
        )

    audit_results.append({
        "year": year,
        "observations": total_rows,
        "unique_dates": len(daily_counts),
        "min_daily_observations": daily_counts.min(),
        "max_daily_observations": daily_counts.max(),
        "null_values": null_values,
        "negative_values": negative_values,
        "min_rainfall": min_rainfall,
        "max_rainfall": max_rainfall
    })

audit_df = pd.DataFrame(audit_results)

display(audit_df)

,year,observations,unique_dates,min_daily_observations,max_daily_observations,null_values,negative_values,min_rainfall,max_rainfall
0,1951,1811860,365,4964.0,4964.0,0,0,0.0,359.30500
1,1952,1816824,366,4964.0,4964.0,0,0,0.0,426.84060
2,1953,1811860,365,4964.0,4964.0,0,0,0.0,412.97920
3,1954,1811860,365,4964.0,4964.0,0,0,0.0,403.06320
4,1955,1811860,365,4964.0,4964.0,0,0,0.0,443.29510
...,...,...,...,...,...,...,...,...,...
70,2021,1811860,365,4964.0,4964.0,0,0,0.0,526.71060
71,2022,1811860,365,4964.0,4964.0,0,0,0.0,979.14480
72,2023,1811781,365,4885.0,4964.0,0,0,0.0,689.03723
73,2024,1816824,366,4964.0,4964.0,0,0,0.0,660.54920


In [2]:
# Inspect daily coverage in 2023
df_2023 = pd.read_csv(
    INTERIM_DIR / "imd_rainfall_2023.csv",
    parse_dates=["date"]
)

daily_counts_2023 = df_2023.groupby("date").size()

print("2023 days with fewer than 4964 observations:")
print(daily_counts_2023[daily_counts_2023 < 4964])

# Inspect the highest rainfall observations in 2022
df_2022 = pd.read_csv(
    INTERIM_DIR / "imd_rainfall_2022.csv"
)

print("\nTop 10 rainfall observations in 2022:")
display(
    df_2022.nlargest(10, "rainfall_mm")
)

2023 days with fewer than 4964 observations:
date
2023-10-12    4885
dtype: int64

Top 10 rainfall observations in 2022:


,date,latitude,longitude,rainfall_mm
831948,2022-06-17,25.25,91.25,979.14480
831946,2022-06-17,25.25,90.75,921.06024
831947,2022-06-17,25.25,91.00,919.28180
832040,2022-06-17,25.50,90.75,896.86050
831949,2022-06-17,25.25,91.50,891.51355
832041,2022-06-17,25.50,91.00,862.29400
822021,2022-06-15,25.25,91.50,779.41680
822018,2022-06-15,25.25,90.75,753.87180
832042,2022-06-17,25.50,91.25,751.10110
822020,2022-06-15,25.25,91.25,747.54720


In [3]:
# Inspect the highest rainfall observations across the entire 1951–2025 period

top_extremes = []

for year in range(1951, 2026):

    file_path = INTERIM_DIR / f"imd_rainfall_{year}.csv"

    for chunk in pd.read_csv(file_path, chunksize=500_000):

        top = chunk.nlargest(5, "rainfall_mm")

        top_extremes.append(top)

# Combine the candidate extreme observations
top_extremes_df = pd.concat(top_extremes)

# Get the overall top 20
top_20 = top_extremes_df.nlargest(20, "rainfall_mm")

display(top_20.reset_index(drop=True))

,date,latitude,longitude,rainfall_mm
0,2022-06-17,25.25,91.25,979.14480
1,1982-08-04,25.25,91.25,939.44617
2,1982-07-06,25.25,91.25,938.15234
3,2022-06-17,25.25,90.75,921.06024
4,1982-08-03,25.25,91.25,920.22687
5,2022-06-17,25.25,91.00,919.28180
6,1974-09-13,25.25,91.25,898.93500
7,2022-06-17,25.50,90.75,896.86050
8,2022-06-17,25.25,91.50,891.51355
9,1974-09-12,25.25,91.25,882.28235


In [4]:
# Rainfall distribution on 17 June 2022
df_2022_event = df_2022[
    df_2022["date"] == "2022-06-17"
].copy()

print("Grid cells with rainfall above 100 mm:",
      (df_2022_event["rainfall_mm"] > 100).sum())

print("Grid cells with rainfall above 200 mm:",
      (df_2022_event["rainfall_mm"] > 200).sum())

print("\nRainfall statistics for the day:")
print(df_2022_event["rainfall_mm"].describe())

Grid cells with rainfall above 100 mm: 80
Grid cells with rainfall above 200 mm: 24

Rainfall statistics for the day:
count    4964.000000
mean       10.892551
std        41.293253
min         0.000000
25%         0.000000
50%         0.822373
75%         8.291187
max       979.144800
Name: rainfall_mm, dtype: float64


In [5]:
#checking for duplicate values

duplicate_results = []

for year in range(1951, 2026):

    file_path = INTERIM_DIR / f"imd_rainfall_{year}.csv"

    df_year = pd.read_csv(file_path)

    duplicates = df_year.duplicated(
        subset=["date", "latitude", "longitude"]
    ).sum()

    duplicate_results.append({
        "year": year,
        "duplicate_observations": duplicates
    })

duplicates_df = pd.DataFrame(duplicate_results)

display(duplicates_df)

print(
    "\nTotal duplicate observations:",
    duplicates_df["duplicate_observations"].sum()
)

,year,duplicate_observations
0,1951,0
1,1952,0
2,1953,0
3,1954,0
4,1955,0
...,...,...
70,2021,0
71,2022,0
72,2023,0
73,2024,0



Total duplicate observations: 0


In [7]:
# checking if grid coverage across the country is same throughout the years

reference_df = pd.read_csv(
    INTERIM_DIR / "imd_rainfall_1901.csv",
    usecols=["latitude", "longitude"]
)

reference_grid = set(
    zip(reference_df["latitude"], reference_df["longitude"])
)

print("Reference grid cells in 1901:", len(reference_grid))


# Compare all years from 1901 to 2025
coverage_results = []

for year in range(1901, 2026):

    file_path = INTERIM_DIR / f"imd_rainfall_{year}.csv"

    df_year = pd.read_csv(
        file_path,
        usecols=["latitude", "longitude"]
    )

    current_grid = set(
        zip(df_year["latitude"], df_year["longitude"])
    )

    coverage_results.append({
        "year": year,
        "unique_grid_cells": len(current_grid),
        "missing_vs_1901": len(reference_grid - current_grid),
        "additional_vs_1901": len(current_grid - reference_grid)
    })

coverage_df = pd.DataFrame(coverage_results)

display(coverage_df)

coverage_df = pd.DataFrame(coverage_results)

different_years = (
    (coverage_df["missing_vs_1901"] > 0) |
    (coverage_df["additional_vs_1901"] > 0)
).sum()

print("Total years checked:", len(coverage_df))
print("Years with different geographical coverage:", different_years)
print("Years with identical geographical coverage:", len(coverage_df) - different_years)

Reference grid cells in 1901: 4964


,year,unique_grid_cells,missing_vs_1901,additional_vs_1901
0,1901,4964,0,0
1,1902,4964,0,0
2,1903,4964,0,0
3,1904,4964,0,0
4,1905,4964,0,0
...,...,...,...,...
120,2021,4964,0,0
121,2022,4964,0,0
122,2023,4964,0,0
123,2024,4964,0,0


Total years checked: 125
Years with different geographical coverage: 0
Years with identical geographical coverage: 125
